In [7]:
import torch
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import struct

In [15]:
def read_idx_images(filename):
    with open(filename, 'rb') as f:
        _, _, dims = struct.unpack('>HBB', f.read(4))
        shape = tuple(struct.unpack('>I', f.read(4))[0] for _ in range(dims))
        data = np.frombuffer(f.read(), dtype=np.uint8).reshape(shape)
    return data

def read_idx_labels(filename):
    with open(filename, 'rb') as f:
        _, num_items = struct.unpack('>II', f.read(8))
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data

def load_data(images_path, labels_path, num_classes=10):
    # Read images and labels
    images = read_idx_images(images_path).astype(np.float32) / 255.0  # normalize to 0–1
    labels = read_idx_labels(labels_path)

    # Flatten images (N, 28*28)
    images = images.reshape(images.shape[0], -1)

    # One-hot encode labels
    labels_onehot = np.eye(num_classes)[labels].astype(np.float32)

    # Convert to torch tensors
    X = torch.tensor(images)
    y = torch.tensor(labels_onehot)

    return X, y

In [30]:
train_images = "train-images.idx3-ubyte"
train_labels = "train-labels.idx1-ubyte"
test_images  = "t10k-images.idx3-ubyte"
test_labels  = "t10k-labels.idx1-ubyte"

X_train, y_train = load_data(train_images, train_labels, num_classes=10)
X_test, y_test   = load_data(test_images, test_labels, num_classes=10)

train_dataset = TensorDataset(X_train, y_train)
test_dataset  = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)